<div style="display: flex; background-color: RGB(255,114,0);" >
<h1 style="margin: auto; padding: 30px; ">ANALYSE DU STOCK ET DES VENTES DU SITE BOTTLENECK</h1>
</div>

# **<div style="background-color: RGB(51,165,182);" >**
<h2 style="margin: auto; padding: 20px; color:#fff; ">Etape 1 - Importation des librairies et chargement des fichiers</h2>
</div>

<div style="border: 1px solid RGB(51,165,182);" >
<h3 style="margin: auto; padding: 20px; color: RGB(51,165,182); ">1.1 - Importation des librairies</h3>
</div>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

ModuleNotFoundError: No module named 'google'

In [ ]:
#Importation de la librairie Pandas
import pandas as pd
import numpy as np

#Importation de la librairie plotly express
import plotly.express as px

In [ ]:
pd.set_option('display.max_columns', None)   # afficher toutes les colonnes
pd.set_option('display.width', 1000)         # élargir la largeur d'affichage
pd.set_option('display.max_colwidth', None)  # éviter la coupure du texte
pd.set_option('display.max_rows', None)      # afficher toutes les lignes

<div style="border: 1px solid RGB(51,165,182);" >
<h3 style="margin: auto; padding: 20px; color: RGB(51,165,182); ">1.2 - Chargements des fichiers</h3>
</div>

In [ ]:
#Importation du fichier web.xlsx
df_web = pd.read_excel("/content/drive/MyDrive/Colab Notebooks/web.xlsx")
#Importation du fichier erp.xlsx
df_erp = pd.read_excel("/content/drive/MyDrive/Colab Notebooks/erp.xlsx")
#Importation du fichier liaison.xlsx
df_liaison = pd.read_excel("/content/drive/MyDrive/Colab Notebooks/liaison.xlsx")
# L'avertissement indique que openpyxl n'arrive pas à lire certains éléments qui sont ignorés (sans incidences sur le projet)

<div style="background-color: RGB(51,165,182);" >
<h2 style="margin: auto; padding: 20px; color:#fff; ">Etape 2 - Analyse exploratoire des fichiers</h2>
</div>

<div style="border: 1px solid RGB(51,165,182);" >
<h3 style="margin: auto; padding: 20px; color: RGB(51,165,182); ">2.1 - Analyse exploratoire du fichier erp.xlsx</h3>
</div>

In [ ]:
# Methode shape
print(df_erp.shape)

# Afficher les dimensions du dataset
print("Le tableau comporte {} articles".format(df_erp.shape[0]))
print("Le tableau comporte {} colonnes".format(df_erp.shape[1]))

# Autre maniere avec f-strings
print(f"Dimensions : {df_erp.shape[0]} lignes × {df_erp.shape[1]} colonnes")

In [ ]:
# Nature des données et valeurs présentes dans chacune des colonnes
df_erp.info()

In [ ]:
# Afficher les 5 premières lignes de la table
df_erp.head()

In [ ]:
# Vérifier si il y a des lignes en doublon dans la colonne product_id
nb_doublons = df_erp.duplicated('product_id').sum()
print(f"Doublons sur product_id : {nb_doublons}")

# Vérifier que chaque valeur de la colonne product_id est unique
df_erp['product_id'].nunique() == len(df_erp) # Si cette égalité est vraie alors chaque ligne correspond à un id unique => pas de duplication

Les valeurs de la colonne "product_id" permettent d’identifier de façon unique chaque enregistrement de la table "erp". C'est donc une clé primaire.

In [ ]:
#Afficher les valeurs distinctes de la colonne stock_status
print("Valeurs de stock_status :")
print(df_erp['stock_status'].value_counts()) # Analyse de la distribution des valeurs catégorielles de la colonne 'Stock_status'

La colonne stock_status doit être en cohérence avec les valeurs de la colonne stock_quantity.

In [ ]:
# Création d'une colonne stock_status_2 calculée à partir de stock_quantity : si la valeur de la colonne "stock_quantity" est nulle, renseigner "outofstock" sinon mettre "instock"
df_erp['stock_status_2'] = df_erp['stock_quantity'].apply(
    lambda x:
        'instock' if x > 0 else 'outofstock'
)
# Verification
df_erp.head()

In [ ]:
# Comparaison des deux colonnes: on considere "stock_status_2" comme la source de vérité
nb_coherents = (df_erp['stock_status'] == df_erp['stock_status_2']).sum()
nb_incoherents = (df_erp['stock_status'] != df_erp['stock_status_2']).sum()

print(f"Lignes cohérentes   : {nb_coherents}")
print(f"Lignes incohérentes : {nb_incoherents}")

🔴 ERREUR N°1 — Incohérence stock_status / stock_quantity

La colonne stock_status (instock / outofstock) doit refléter la colonne stock_quantity.

In [ ]:
# Création d'un mask anomalies : si l'inégalité est vraie elle renvoie le booléen True (True=1)
mask = df_erp['stock_status'] != df_erp['stock_status_2']

# Affichage des lignes en écart
if nb_incoherents > 0:
    print("Lignes avec incohérence :")
    print(df_erp[mask][['product_id', 'stock_quantity', 'stock_status', 'stock_status_2']])

In [ ]:
# Correction : on remplace les valeurs par celles données par la source de vérité (stock_status_2)
df_erp['stock_status'] = df_erp['stock_status_2']
df_erp.drop(columns=['stock_status_2'], inplace=True) # On supprime la colonne 'stock_status_2'

<div style="border: 1px solid RGB(51,165,182);" >
<h3 style="margin: auto; padding: 20px; color: RGB(51,165,182); ">2.1.1 - Analyse exploratoire de chaque variable du fichier erp.xlsx</h3>
</div>

<div style="border: 1px solid RGB(51,165,182);" >
<h3 style="margin: auto; padding: 20px; color: RGB(51,165,182); ">2.1.1.1 - Analyse de la variable PRIX</h3>
</div>

In [ ]:
###############
## LES PRIX  ##
###############

# Afficher le ou les prix non renseignés dans la colonne "price"
print(f"Prix manquants  : {df_erp['price'].isna().sum()}") # isna détecte les valeurs manquantes (NaN), sum compte les 'True'
print("Nombres d'articles avec un prix non renseigné: {}".format(df_erp['price'].isna().sum())) # autre méthode

# Afficher le prix minimum de la colonne "price"
print(f"Prix minimum    : {df_erp['price'].min()}")

# Afficher le prix maximum de la colonne "price"
print(f"Prix maximum    : {df_erp['price'].max()}")

# Afficher les prix inférieurs à 0 => valeurs aberrantes
prix_negatifs = df_erp[df_erp['price'] < 0]
print(f"Articles avec prix négatif : {len(prix_negatifs)}")
print(prix_negatifs[['product_id', 'price', 'purchase_price', 'stock_quantity']])

# Afficher les prix nuls
prix_nuls = df_erp[df_erp['price'].isnull()]
print("Nombres d'articles avec un prix nul: {}".format(df_erp['price'].isnull().sum()))


🔴 ERREUR N°2 — Prix négatifs dans l'ERP

Des prix négatifs sont impossibles pour un article en vente. Il s'agit très probablement d'erreurs de saisie.

In [ ]:
# Correction : on transforme les valeurs de 'price' en valeur absolue pour ne plus avoir de prix négatifs
df_erp['price'] = df_erp['price'].abs()
print("✅ Prix négatifs corrigés (valeur absolue appliquée)")
print(f"Nouveau prix minimum : {df_erp['price'].min()}")

<div style="border: 1px solid RGB(51,165,182);" >
<h3 style="margin: auto; padding: 20px; color: RGB(51,165,182); ">2.1.1.2 - Analyse de la variable STOCK</h3>
</div>

In [ ]:

#######################
### stock_quantity  ###
#######################

# Afficher la quantité minimum de la colonne "stock_quantity"
print(f"Stock minimum : {df_erp['stock_quantity'].min()}")

# Afficher la quantité maximum de la colonne "stock_quantity"
print(f"Stock maximum : {df_erp['stock_quantity'].max()}")

# Afficher les stocks inférieurs à 0
stock_negatif = df_erp[df_erp['stock_quantity'] < 0]
print(f"Articles avec stock négatif : {len(stock_negatif)}")
print(stock_negatif[['product_id', 'stock_quantity', 'stock_status', 'price']])

🔴 ERREUR N°3 — Quantités de stock négatives

Un stock ne peut pas être négatif. Ces valeurs correspondent probablement à des erreurs de saisie.

In [ ]:
# Correction : ramener les stocks négatifs à 0 grace à la fonction clip
df_erp['stock_quantity'] = df_erp['stock_quantity'].clip(lower=0) #Si une valeur de stock est inférieure à 0, on la remplace par 0
print("✅ Stocks négatifs ramenés à 0")

<div style="border: 1px solid RGB(51,165,182);" >
<h3 style="margin: auto; padding: 20px; color: RGB(51,165,182); ">2.1.1.3 - Analyse de la variable ONSALE_WEB</h3>
</div>

In [ ]:
# La colonne onsale_web designe les ventes réalisées sur le web "onsale_web == 1"
print(f"Il y a {(df_erp['onsale_web']==1).sum()} références produit vendues en ligne.")

In [ ]:
# Selection des ventes réalisées exclusivement sur le web
df_erp = df_erp[df_erp['onsale_web']==1].copy()
df_erp.info()

Pour la suite de l'analyse on conserve le dataframe df_erp filtré sur les ventes en ligne.

<div style="border: 1px solid RGB(51,165,182);" >
<h3 style="margin: auto; padding: 20px; color: RGB(51,165,182); ">2.1.1.4 - Analyse de la variable prix d'achat</h3>
</div>

In [ ]:
######################
##   prix d'achat   ##
######################

# Afficher le ou les prix non renseignés dans la colonne "purchase_price"
print(f"Prix d'achat non renseignés  : {df_erp['purchase_price'].isna().sum()}") # isna détecte les valeurs manquantes (NaN), sum compte les True

# Afficher le prix maximum de la colonne "purchase_price"
print(f"Prix d'achat maximum : {df_erp['purchase_price'].max()}")

# Afficher le prix minimum de la colonne "purchase_price"
print(f"Prix d'achat minimum : {df_erp['purchase_price'].min()}")

In [ ]:
# Comparaison du prix d'achat et de vente
df_achat_vente = df_erp.copy()
df_achat_vente['marge'] = df_achat_vente['price']-df_achat_vente['purchase_price']
df_achat_vente.loc[df_achat_vente['marge']<0,:]

🔴 ERREUR N°4 — Prix de vente inférieur au prix d'achat

Une marge négative identifiée sur un produit indique une incohérence entre le prix de vente et le prix d’achat. Cette anomalie est probablement liée à une erreur de saisie ou à un problème de mise à jour des données. Afin de garantir la fiabilité de l’analyse, cette observation est exclue du dataset.

In [ ]:
# On exclut la valeur incohérente
df_achat_vente.loc[df_achat_vente['product_id'] == 4355, ['price', 'purchase_price']] = np.nan

In [ ]:
# On enregistre les modifications dans le dataframe df_erp
df_erp = df_achat_vente.copy()

In [ ]:
df_erp.loc[df_erp['product_id'] == 4355, :]

In [ ]:
df_erp.info()


<div style="border: 1px solid RGB(51,165,182);" >
<h3 style="margin: auto; padding: 20px; color: RGB(51,165,182); ">2.2 - Analyse exploratoire du fichier web.xlsx</h3>
</div>


In [ ]:
# Dimension du dataset
print(df_web.shape)

# Nombre d'observations
print("Le tableau comporte {} observations\n".format(df_web.shape[0]))

# Nombre de caractéristiques
print("Le tableau comporte {} colonnes\n".format(df_web.shape[1]))

# Autre maniere avec f-strings
print(f"Dimensions : {df_web.shape[0]} lignes × {df_web.shape[1]} colonnes\n")

In [ ]:
# La nature des données dans chacune des colonnes et le nombre de valeurs présentes dans chacune des colonnes
resume = pd.DataFrame({
    "Type": df_web.dtypes,
    "Valeurs_non_nulles": df_web.count(),
    "Valeurs_manquantes": df_web.isnull().sum(),
    "Total_lignes": len(df_web)
})

print(resume)

In [ ]:
# Suppression des colonnes sans valeur métier
cols_to_drop = [
    "tax_class", "post_content", "post_password", "post_content_filtered","guid", "menu_order", "post_mime_type", "ping_status","post_date_gmt", "post_modified_gmt",
    "post_excerpt", "post_name", "virtual", "downloadable", "post_parent", "comment_status", "comment_count", "rating_count", "average_rating", "tax_status", "post_author", "post_status",
    "post_modified", "post_date"
]

df_web = df_web.drop(columns=cols_to_drop, errors="ignore")


In [ ]:
df_web.head()

In [ ]:
print("Vérification des SKU qui ne semblent pas respecter la régle de codification")

# SKU manquants
mask_null = df_web['sku'].isna()
nb_null = df_web['sku'].isna().sum()

# SKU en doublon
mask_dup = df_web['sku'].duplicated()
nb_dup = df_web['sku'].duplicated().sum()

# Regroupement des anomalies
mask_anomalies = mask_null | mask_dup

# Consultation des lignes problématiques
df_anomalies = df_web[mask_anomalies][:]
print(f"SKU manquants : {nb_null}")
print(f"SKU en doublon : {nb_dup}")
print(f"Total anomalies : {len(df_anomalies)}")

In [ ]:
# Lignes sans code article
df_sans_sku = df_web[
    df_web["sku"].isnull() | (df_web["sku"].str.strip() == "") # sku manquants OU chaînes vides/remplies d’espaces
]
print(f"Il y a {len(df_sans_sku)} lignes sans code article sku.\n")
df_sans_sku.info()

🔴 ERREUR N°5 — Les lignes sans code article ne peuvent pas être exploitées.


In [ ]:
# Visualisation des lignes non nulles dans le dataframe df_sans_sku
df_sans_sku[df_sans_sku["total_sales"].notnull()]

🔴 ERREUR N°6 — Total des ventes (total_sales) négatif

Des ventes négatives n'ont pas de sens commercial. Cela n'a pas d'incidence car ces lignes font parties des lignes sans sku supprimées.


In [ ]:
# Exclusion des lignes sans code article SKU
df_web_clean = df_web[
    ~(df_web["sku"].isnull() | (df_web["sku"].str.strip() == "")) # garder uniquement les lignes où le code article (sku) est valide
]

In [ ]:
# On verifie bien passer de 1513 lignes à 1428 soit 85 de moins
df_web.info()
df_web_clean.info()

In [ ]:
# Identification des doublons
df_doublons = df_web_clean[df_web_clean["sku"].duplicated(keep=False)]

In [ ]:
# Visualisation des doublons
for i, (sku, group) in enumerate(df_doublons.groupby("sku")):
    if i == 5:
        break
    print(f"\n=== SKU : {sku} ===")
    display(group)

In [ ]:
# Identifier les différents types de la colonne "post_type"
print("Types de post_type :")
print(df_web_clean['post_type'].value_counts())

In [ ]:
# On conserve uniquement les produits de la colonne "post_type"
df_web_products = df_web_clean[df_web_clean['post_type'] == 'product'].copy()
print(f"Produits en ligne : {len(df_web_products)}")

In [ ]:
df_web_products.info()

In [ ]:
# Identifier les différents types de la colonne 'SKU' (car la fonction info indique un melange de type "objet")
df_web_products['sku'].apply(type).value_counts()

In [ ]:
# Identifier les valeurs suspects c-à-dire non 'int' (montrer les valeurs qui n’ont pas pu être converties en nombre) => isna()
df_sku_non_numeric = df_web_products[
    pd.to_numeric(df_web_products['sku'], errors='coerce').isna()
]
df_sku_non_numeric

In [ ]:
df_sku_numeric = df_web_products[
    pd.to_numeric(df_web_products['sku'], errors='coerce').notna()
].copy()

df_sku_numeric['sku'] = df_sku_numeric['sku'].astype(int) # Conversion en type int afin d'assurer la jointure avec 'id_web' (cf.df_liaison)

🔴 ERREUR N°7 — Les valeurs non numériques de la clé sku ont été identifiées via une conversion contrôlée (pd.to_numeric). Les valeurs invalides, transformées en NaN (errors='coerce'), ont été isolées avec isna() puis exclues du dataset avec notna() afin de garantir la fiabilité des jointures.

In [ ]:
df_web_products = df_sku_numeric
df_web_products.info()

In [ ]:
df_web_products.head()

<div style="border: 1px solid RGB(51,165,182);" >
<h3 style="margin: auto; padding: 20px; color: RGB(51,165,182); ">2.3 - Analyse exploratoire du fichier liaison.xlsx</h3>
</div>

In [ ]:
#Dimension du dataset
print(df_liaison.shape)

#Nombre d'observations
print("Le tableau comporte {} observations\n".format(df_liaison.shape[0]))

#Nombre de caractéristiques
print("Le tableau comporte {} colonnes\n".format(df_liaison.shape[1]))

df_liaison.head()

In [ ]:
# La nature des données dans chacune des colonnes et le nombre de valeurs présentes dans chacune des colonnes
resume_liaison = pd.DataFrame({
    "Type": df_liaison.dtypes,
    "Valeurs_non_nulles": df_liaison.count(),
    "Valeurs_manquantes": df_liaison.isnull().sum(),
    "Total_lignes": len(df_liaison)
})

print(resume_liaison)

In [ ]:
# Vérifier que chaque valeur de la colonne "id_web" est unique
is_unique = df_liaison["id_web"].is_unique
print("id_web est unique ?", is_unique)


In [ ]:
# Compter les doublons (hors NaN)
nb_doublons = df_liaison[
    df_liaison['id_web'].notna()
]['id_web'].duplicated().sum()

print(nb_doublons)

In [ ]:
# Détail des différents type de la colonne id_web
df_liaison['id_web'].apply(type).value_counts()

In [ ]:
# Identification des valeurs "id_web" non numériques
df_liaison[df_liaison['id_web'].apply(lambda x: isinstance(x, str))]

🔴 ERREUR N°8 — Identification de 3 valeurs "id_web" non numérique

In [ ]:
# Vérifier si il y a des lignes en doublon dans la colonne "product_id"
nb_doublons = df_liaison.duplicated('product_id').sum()
print(f"Doublons sur product_id : {nb_doublons}")

# Vérifier que chaque valeur de la colonne product_id est unique
df_liaison['product_id'].nunique() == len(df_liaison) # Si cette égalité est vraie alors chaque ligne correspond à un id unique => pas de duplication

In [ ]:
# Vérifier les articles ERP sans correspondance Web
df_sans_web = df_liaison[df_liaison["id_web"].isnull()]
print("Articles sans correspondance Web :", len(df_sans_web))
df_sans_web.head()

🔴 ERREUR N°9 — Produits ERP sans correspondance web (id_web manquant)

La table de liaison contient des product_id ERP sans id_web associé. Ces produits existent dans l'ERP mais ne sont pas publiés sur le site web.

In [ ]:
# Vérifier les articles Web sans correspondance ERP
df_sans_erp = df_liaison[df_liaison["product_id"].isnull()]
print("Articles sans correspondance ERP :", len(df_sans_erp))
df_sans_erp.head()

A contrario dans la table de liaison tous les articles WEB ont une correspondance dans ERP.

In [ ]:
# Création d'une table de liaison fiable
df_liaison_clean = df_liaison.copy()

# Conversion sécurisée en numérique
df_liaison_clean['id_web'] = pd.to_numeric(df_liaison_clean['id_web'], errors='coerce')

# Garder uniquement les lignes valides
df_liaison_clean = df_liaison_clean[
    df_liaison_clean['id_web'].notna()
].copy()

# Conversion finale en entier
df_liaison_clean['product_id'] = df_liaison_clean['product_id'].astype(int)
df_liaison_clean['id_web'] = df_liaison_clean['id_web'].astype(int)

In [ ]:
# Table de liaison propre
print(f"Lignes retenues dans la table de liaison : {len(df_liaison_clean)}")

In [ ]:
df_liaison_clean.info()

<div style="background-color: RGB(51,165,182);" >
<h2 style="margin: auto; padding: 20px; color:#fff; ">Etape 3 - Jonction des fichiers</h2>
</div>

<div style="border: 1px solid RGB(51,165,182);" >
<h3 style="margin: auto; padding: 20px; color: RGB(51,165,182); ">Etape 3.1 - Jonction du fichier df_erp et df_liaison</h3>
</div>

In [ ]:
# Fusion des fichiers df_erp et df_liaison
# On se base sur les 716 références produit de df_erp vendues sur le web (c-à-dire 'onsale_web' == 1)
df_merge = pd.merge(df_erp, df_liaison_clean, on ='product_id', how='left')
df_merge.info()

In [ ]:
# Identifier les références produit présentent dans l'ERP mais qui ne sont pas mappées dans la table de liaison
df_sans_liaison = df_erp[
    ~df_erp["product_id"].isin(df_liaison_clean["product_id"])
]
df_sans_liaison

🔴 ERREUR N°10 — Certaines références produit normalement vendues sur le web ne sont pas mappées dans la table de liaison.

<div style="border: 1px solid RGB(51,165,182);" >
<h3 style="margin: auto; padding: 20px; color: RGB(51,165,182); ">Etape 3.2 - Jonction du fichier df_merge et df_web</h3>
</div>

In [ ]:
# Fusion des datasets df_merge et df_web_products
df_merge_web = pd.merge(df_merge, df_web_products, left_on='id_web', right_on='sku', how='left')
df_merge_web.head()

In [ ]:
# Colonnes utiles pour l'analyse
colonnes_utiles = ['product_id','id_web','product_type', 'post_title',
                   'total_sales', 'price', 'purchase_price',
                   'stock_quantity', 'stock_status']
df_final = df_merge_web[colonnes_utiles].copy()
df_final.info()
df_final.head()

In [ ]:
# Supprimer les lignes avec id_web manquant
df_final = df_final[df_final["id_web"].notna()].copy()

# Conversion des types
df_final["id_web"] = df_final["id_web"].astype(int)
df_final["total_sales"] = df_final["total_sales"].astype(int)

# Vérification
df_final.info()

Le df_final qui servira à l'analyse comporte 711 observations.

<div style="background-color: RGB(51,165,182);" >
<h2 style="margin: auto; padding: 20px; color:#fff; ">Etape 4 - Analyse univariée des prix</h2>
</div>

<div style="border: 1px solid RGB(51,165,182);" >
<h3 style="margin: auto; padding: 20px; color: RGB(51,165,182); ">Etape 4.1 - Exploration par la visualisation de données</h3>
</div>

In [ ]:
# Création d'une boîte à moustache de la répartition des prix grâce à Pandas
import matplotlib.pyplot as plt
df_final["price"].plot.box(figsize=(8,5))


plt.title("Boîte à moustaches de la répartition des prix")
plt.ylabel("Prix")
plt.show()

In [ ]:
# Autre méthode avec plotly express

fig = px.box(
    df_final,
    y="price",
    title="Boîte à moustaches de la répartition des prix"
)

fig.show()

La moitié des prix est comprise entre 14 € et 42 €.
Le prix médian est de 23,45 €.
La distribution est asymétrique à droite : quelques produits sont nettement plus chers que la majorité.
Il existe des valeurs atypiques élevées.

<div style="border: 1px solid RGB(51,165,182);" >
<h3 style="margin: auto; padding: 20px; color: RGB(51,165,182); ">Etape 4.2 - Exploration par l'utilisation de méthodes statistiques</h3>
</div>

<div style="border: 1px solid RGB(51,165,182);" >
<h3 style="margin: auto; padding: 20px; color: RGB(51,165,182); ">Etape 4.2.1 - Identification par le Z-index</h3>
</div>

In [ ]:
#Calcule de la moyenne du prix
prix_moyen = df_final['price'].mean()
print(f"La moyenne des prix est de {round(prix_moyen,2)} €\n")

#Calculer l'écart-type du prix
ecart_type_prix = df_final['price'].std()
print(f"L'écart-type du prix est de {round(ecart_type_prix,2)} \n")

In [ ]:
# Le Z-score mesure l’écart à la moyenne en nombre d’écarts-types
df_z = df_final # Création du dataframe d'analyse du Z-score
df_z["z_score"] = (
    df_z["price"] - prix_moyen
) / ecart_type_prix

df_z[["price", "z_score"]].head()

Le Z-score permet d'identifier les valeurs atypiques (outliers)
👉 règle classique :
|Z| > 2 → suspect
|Z| > 3 → atypique fort

Dans le dataframe "df_final" Z-score permet de détecter les produits premium et d'identifier le cœur de gamme (Z proche de 0)

In [ ]:
# Visualisation des outliers grace à un graphique en nuage de point :
plt.scatter(df_z.index, df_z["price"], c=df_z["z_score"])
plt.colorbar(label="Z-score")
plt.title("Détection des valeurs atypiques (Z-score)")
plt.show()

In [ ]:
# Calcul du seuil de prix dont le z-score est supérieur à 3 :
seuil = prix_moyen + 3 * ecart_type_prix
print(f"le seuil de prix (Z-score > 3) est de {round(seuil,2)} €")

In [ ]:
# Identification des produits haut de gamme :
df_outliers = df_z[df_z["price"] > seuil]
print("Nombre de produits haut de gamme (Z-score > 3) :", len(df_outliers))
df_outliers[["post_title", "price"]].sort_values("price", ascending=False)

<div style="border: 1px solid RGB(51,165,182);" >
<h3 style="margin: auto; padding: 20px; color: RGB(51,165,182); ">Etape 4.2.2 - Identification par l'intervalle interquartile (IQR) </h3>
</div>

In [ ]:
# Utilisation de la fonction "describe" de Pandas pour l'étude des mesures de dispersion :
stats = round(df_final["price"].describe(),2)
print(stats)

Définition du seuil des outliers par la méthode IQR (écart interquartile) :
👉 IQR = Q3−Q1
👉 Seuil haut = Q3+1.5×IQR
👉 Seuil bas = Q1-1.5×IQR

In [ ]:
Q1 = df_final["price"].quantile(0.25)
Q3 = round(df_final["price"].quantile(0.75),2)

IQR = round(Q3 - Q1,2)

borne_sup = Q3 + 1.5 * IQR
borne_inf = round(Q1 - 1.5 * IQR,2)

print("Q1 :", Q1)
print("Q3 :", Q3)
print("IQR :", IQR)
print("Seuil inférieur :", borne_inf)
print("Seuil supérieur :", borne_sup)

In [ ]:
# Définition du nombre d'articles et proportion du catalogue "outliers"
outliers = df_final[df_final["price"] > borne_sup]

nb_outliers = len(outliers)
nb_total = len(df_final)

proportion = nb_outliers*100 / nb_total

print("Nombre d'outliers :", nb_outliers)
print(f"La proportion des outliers est de {round(proportion,2)} %")

L’analyse par l’intervalle interquartile met en évidence 31 produits atypiques, représentant environ 4,6 % du catalogue. Ces produits présentent des prix supérieurs à 84 €, seuil défini par la méthode IQR.

Ces valeurs ne semblent pas aberrantes mais reflètent plutôt la présence de produits haut de gamme dans le catalogue. La distribution des prix est asymétrique à droite, ce qui est cohérent avec un marché comprenant une majorité de produits standards et une minorité de produits premium.

In [ ]:
# Analyse par type de produit
df_final.groupby("product_type")["price"].describe()

In [ ]:
# Visualiser les produits outliers
df_outliers = outliers[["post_title", "price"]].sort_values("price", ascending=False)
df_outliers.head()

In [ ]:
# Analyse distribution
df_final["price"].hist(bins=50)

plt.title("Distribution des prix")
plt.xlabel("Prix (€)")
plt.ylabel("Nombre de produits")

plt.show()

<div style="background-color: RGB(51,165,182);" >
<h2 style="margin: auto; padding: 20px; color:#fff; ">Etape 5 - Analyse univariée du CA, des quantités vendues, des stocks et de la marge ainsi qu'une analyse multivariée  </h2>
</div>

<div style="border: 1px solid RGB(51,165,182);" >
<h3 style="margin: auto; padding: 20px; color: RGB(51,165,182); ">Etape 5.1 - Analyse des ventes en CA</h3>
</div>

In [ ]:
##############################
# Calculer le CA du site web #
##############################

df_ca = df_final
df_ca['ca_par_article'] = df_ca['price'] * df_ca['total_sales']
df_ca[['post_title', 'price', 'total_sales', 'ca_par_article']].head()

In [ ]:
# Calcul du chiffre d'affaire du site web
ca_total = df_ca['ca_par_article'].sum()
ca_format = f"{ca_total:,.2f}".replace(",", " ").replace(".", ",")
print(f"Le chiffre d'affaires total du site est de {ca_format} €")

In [ ]:
###############################
# Palmarès des articles en CA #
###############################

# Tri dans l'ordre décroissant du CA du dataset
df_sorted = df_final.sort_values(by="ca_par_article", ascending=False)

# Réinitialisation de l'index du dataset :
df_sorted = df_sorted.reset_index(drop=True)
df_sorted.head()

In [ ]:
# Selection et affichage des 20 premiers articles en CA
top20 = df_sorted.head(20).copy()

# Arrondir le chiffre d'affaires pour un affichage plus lisible
top20["ca_par_article"] = top20["ca_par_article"].round().astype(int)

top20[["post_title", "ca_par_article"]]

In [ ]:
# Création du graphique en barres horizontales
fig = px.bar(
    top20,
    x="ca_par_article",
    y="post_title",
    orientation="h",
    text="ca_par_article",  # afficher les valeurs du CA/barre
    title="Top 20 des articles par chiffre d'affaires"
)

# Amélioration de l'affichage
fig.update_traces(textposition="outside")

fig.update_layout(
    xaxis_title="Chiffre d'affaires (€)",
    yaxis_title="Produits",
    height=800,  # plus d'espace vertical
    width=1150,
    margin=dict(l=300),  # espace pour les noms longs
    yaxis=dict(autorange="reversed", automargin=True),
    template="plotly_white"
)

# Affichage du graphique
fig.show()

In [ ]:
#############################
# Calculer le 20 / 80 en CA #
#############################

# Trie du dataset par chiffre d'affaires décroissant et réinitialisation de l'index
df_pareto = df_final.sort_values(by="ca_par_article", ascending=False).reset_index(drop=True).copy()

# Calculer le chiffre d'affaires total
ca_total = df_pareto["ca_par_article"].sum()

In [ ]:
# Création d'une colonne contenant la part du CA de chaque article dans le CA total
df_pareto["part_ca"] = df_pareto["ca_par_article"] / ca_total

In [ ]:
# Création d'une colonne contenant la somme cumulative de la part de CA
df_pareto["part_ca_cumulee"] = df_pareto["part_ca"].cumsum()

In [ ]:
# Identification des articles qui représentent 80 % du CA
articles_80 = df_pareto[df_pareto["part_ca_cumulee"] <= 0.80]

In [ ]:
# Calcule du nombre d'articles représentant 80 % du CA
nb_articles_80 = len(articles_80)

In [ ]:
# Calcule de la proportion de ces articles dans le catalogue complet
nb_total_articles = len(df_pareto)
proportion_articles_80 = nb_articles_80 / nb_total_articles

In [ ]:
# Affichage des résultats :
print("CA total :", ca_format, "€")
print("Nombre d'articles représentant 80 % du CA :", nb_articles_80)
print("Nombre total d'articles dans le catalogue :", nb_total_articles)
print("Proportion dans le catalogue :", round(proportion_articles_80 * 100, 2), "%")

# Affichage d'un aperçu des colonnes créées :
display(df_pareto[["post_title", "ca_par_article", "part_ca", "part_ca_cumulee"]].head(10))

L’analyse de Pareto montre que le chiffre d’affaires est concentré sur un nombre limité d’articles. En calculant la part de CA de chaque produit puis son cumul, on identifie le nombre d’articles nécessaires pour atteindre 80 % du chiffre d’affaires total. Cela permet de mettre en évidence les produits stratégiques du catalogue : 60% des articles constituent 80% du CA.

<div style="border: 1px solid RGB(51,165,182);" >
<h3 style="margin: auto; padding: 20px; color: RGB(51,165,182); ">Etape 5.2 - Analyse des ventes en quantité</h3>
</div>

In [ ]:
#####################################
# Palmarès des articles en quantité #
#####################################

# Tri dans l'ordre décroissant de quantités vendues
top20_qty = df_final.sort_values(by='stock_quantity', ascending=False)

# Réinitialisation de l'index
top20_qty = top20_qty.reset_index(drop=True)

# Selection des 20 premiers articles en quantité
top20_qty = top20_qty.head(20).copy()

In [ ]:
# Graphique en barre des 20 premiers articles avec plotly express

fig = px.bar(
    top20_qty,
    x="stock_quantity",
    y="post_title",
    orientation="h",
    text="stock_quantity",
    title="Top 20 des articles par quantités vendues"
)

# Amélioration de l'affichage
fig.update_traces(textposition="outside")

fig.update_layout(
    xaxis_title="Quantités vendues",
    yaxis_title="Produits",
    height=800,  # plus d'espace vertical
    width=1000,
    margin=dict(l=300),
    yaxis=dict(autorange="reversed", automargin=True),
    template="plotly_white"
)

fig.show()

In [ ]:
##################################
# Calculer le 20 / 80 en quantité #
##################################

# Trie du dataset par quantité décroissante et réinitialisation de l'index
df_pareto_qty = df_final.sort_values(by='total_sales', ascending=False).reset_index(drop=True).copy()
df_pareto_qty = df_pareto_qty[colonnes_utiles].copy()
df_pareto_qty.head()

In [ ]:
# Créer une colonne de part en quantité
df_pareto_qty["part_qty"] = df_pareto_qty["total_sales"] / df_pareto_qty["total_sales"].sum()

# Créer la somme cumulative de cette part
df_pareto_qty["part_qty_cumulee"] = df_pareto_qty["part_qty"].cumsum()

# Calculer le nombre d'articles représentant 80 % des ventes
nb_articles_80_qty = (df_pareto_qty["part_qty_cumulee"] <= 0.80).sum()

# Calculer la proportion que cela représente dans le catalogue
proportion_catalogue_qty = nb_articles_80_qty / df_pareto_qty["product_id"].nunique()

# Affichage
print(f"Nombre d'articles représentant 80 % des ventes : {nb_articles_80_qty}")
print(f"Proportion du catalogue : {proportion_catalogue_qty:.2%}")


In [ ]:
# Création d'un graphique en nuage de point pour segmenter les produits autour des médianes CA et quantité

# Copie de travail
df_viz = df_final.copy()

# Colonnes utiles
df_viz = df_viz[["post_title", "total_sales", "price"]].dropna().copy()

# Typage
df_viz["total_sales"] = df_viz["total_sales"].astype(int)
df_viz["price"] = df_viz["price"].astype(float)

# Calcul du CA
df_viz["ca_par_article"] = df_viz["price"] * df_viz["total_sales"]

In [ ]:
df_matrix = df_viz.copy()

# Seuils : médianes
x_median = df_matrix["total_sales"].median()
y_median = df_matrix["ca_par_article"].median()

def segment_produit(row):
    if row["total_sales"] >= x_median and row["ca_par_article"] >= y_median:
        return "Stars"
    elif row["total_sales"] < x_median and row["ca_par_article"] >= y_median:
        return "Premium"
    elif row["total_sales"] >= x_median and row["ca_par_article"] < y_median:
        return "Volume"
    else:
        return "Faibles"

df_matrix["segment"] = df_matrix.apply(segment_produit, axis=1)

df_matrix["label"] = np.where(
    df_matrix["ca_par_article"] >= df_matrix["ca_par_article"].quantile(0.97),
    df_matrix["post_title"],
    ""
)

In [ ]:
fig = px.scatter(
    df_matrix,
    x="total_sales",
    y="ca_par_article",
    color="segment",
    hover_data=["post_title", "price"],
    title="Matrice CA vs volume"
)

# Lignes de séparation
fig.add_vline(x=x_median, line_dash="dash")
fig.add_hline(y=y_median, line_dash="dash")

# Noms des quadrants
x_max = df_matrix["total_sales"].max()
y_max = df_matrix["ca_par_article"].max()


fig.update_layout(
    xaxis_title="Quantités vendues",
    yaxis_title="CA par article (€)",
    template="plotly_white"
)

fig.show()

<div style="border: 1px solid RGB(51,165,182);" >
<h3 style="margin: auto; padding: 20px; color: RGB(51,165,182); ">Etape 5.3 - Analyse des stocks</h3>
</div>

In [ ]:
# Création du dataframe df_stock
df_stock = df_final.copy()


In [ ]:
######################################
# Calculer le nombre de mois de stock #
######################################

# Création de la colonne "rotation_stock" c-à-dire le stock disponible / quantités vendues
df_stock['rotation_stock'] = np.where(
    df_stock['total_sales'] > 0,
    (df_stock['stock_quantity'] / df_stock['total_sales']),
    np.nan
).round(1)

# Tri dans l'ordre décroissant du nombre de mois de stock
df_stock_sorted = df_stock.sort_values(by="rotation_stock", ascending=False).reset_index(drop=True)

# Sélection du flop 20 des produits ayant le plus de mois de stock
flop20_stock = df_stock_sorted.head(20).copy()

# Arrondir pour un affichage plus lisible
flop20_stock["rotation_stock"] = flop20_stock["rotation_stock"].round(1)

# Afficher les 20 produits concernés
display(flop20_stock[["post_title", "stock_quantity", "total_sales", "rotation_stock"]])

In [ ]:
# Graphique en barre des 20 premiers articles avec des rotation de stock important

fig = px.bar(
    flop20_stock,
    x="rotation_stock",
    y="post_title",
    orientation="h",
    text="rotation_stock",
    title="Flop 20 des produits avec le plus de mois de stock"
)

# Position du texte
fig.update_traces(textposition="outside")

# Mise en forme pour lisibilité
fig.update_layout(
    xaxis_title="Nombre de mois de stock",
    yaxis_title="Produits",
    height=600,
    width=1200,                   # espace vertical
    margin=dict(l=250),          # espace pour noms longs
    yaxis=dict(autorange="reversed", automargin=True),
    template="plotly_white"
)

fig.show()

In [ ]:
####################################
# Valorisation des stocks en euros #
####################################

print("=== VALEUR DU STOCK ===")

df_stock['stock_value'] = df_stock['stock_quantity'] * df_stock['purchase_price']
stock_total = df_stock['stock_value'].sum()
stock_total = round(stock_total)
print(f"Valeur totale du stock (prix d'achat) : {stock_total} €")
print(f"Produits en rupture de stock           : {(df_stock['stock_status'] == 'outofstock').sum()}")
print(f"Produits en stock                      : {(df_stock['stock_status'] == 'instock').sum()}")
print()

In [ ]:
##############################################
# Valorisation du nombre de produits en stock #
##############################################

print(f"La quantité totale de produits en stock est de {df_stock['stock_quantity'].sum()}.")


<div style="border: 1px solid RGB(51,165,182);" >
<h3 style="margin: auto; padding: 20px; color: RGB(51,165,182); ">Etape 5.4 - Analyse du taux de marge</h3>
</div>

In [ ]:
############################
# Analyse du taux de marge #
############################

# Création du dataframe taux de marge
df_marge = df_stock.copy()

# Création de la colonne prix HT
df_marge["price_ht"] = round(df_marge['price'] / 1.2,2)

# Création de la colonne Taux de marge
df_marge["taux_marge"] = np.where(
    df_marge["purchase_price"] > 0,
    (df_marge["price_ht"] - df_marge["purchase_price"]) / df_marge["purchase_price"],
    np.nan
)

# Afficher le prix minimum de la colonne "taux_marge"
taux_marge_min = df_marge["taux_marge"].min()

# Afficher le prix maximum de la colonne "taux_marge"
taux_marge_max = df_marge["taux_marge"].max()

print("Taux de marge minimum :", round(taux_marge_min * 100, 2), "%")
print("Taux de marge maximum :", round(taux_marge_max * 100, 2), "%")

In [ ]:
# Afficher les lignes avec un taux de marge inférieur à 0
df_marge[df_marge["taux_marge"] < 0][
    ["post_title", "price_ht", "purchase_price", "taux_marge"]
]

In [ ]:
# Création d'un dataframe avec les taux positifs en filtrant uniquement les marges positives
df_marge_positive = df_marge[df_marge["taux_marge"] > 0]

# Afficher le prix minimum de la colonne "taux_marge"
taux_min = df_marge_positive["taux_marge"].min()
print("Taux de marge minimum :", round(taux_min * 100,2))

# Afficher le prix maximum de la colonne "taux_marge"
taux_max = df_marge_positive["taux_marge"].max()
print("Taux de marge maximum :", round(taux_max *100,2))


In [ ]:
# Création d'un dataframe avec le taux de marge moyen par type de produit
df_marge_type = df_marge.groupby("product_type")["taux_marge"].mean().reset_index()

# Conversion en pourcentage
df_marge_type["taux_marge_pct"] = (df_marge_type["taux_marge"] * 100).round(1)

# Tri décroissant
df_marge_type = df_marge_type.sort_values(by="taux_marge_pct", ascending=False)

# Création du graphique
fig = px.bar(
    df_marge_type,
    x="taux_marge_pct",
    y="product_type",
    orientation="h",
    text="taux_marge_pct",
    title="Taux de marge moyen (%) par type de produit",
    color="taux_marge_pct",  # couleur selon valeur
    color_continuous_scale="Blues"
)

# Amélioration des labels
fig.update_traces(
    texttemplate="%{text}%",
    textposition="outside"
)

# Mise en forme globale
fig.update_layout(
    xaxis_title="Taux de marge moyen (%)",
    yaxis_title="Type de produit",
    yaxis=dict(autorange="reversed", automargin=True),
    height=500,
    template="plotly_white",
    coloraxis_colorbar=dict(title="Marge (%)")
)

fig.show()

<div style="border: 1px solid RGB(51,165,182);" >
<h3 style="margin: auto; padding: 20px; color: RGB(51,165,182); ">Etape 5.5 - Analyse des corrélations entre les variables stock, sales et price</h3>
</div>

In [ ]:
############################
# Analyse des corrélations #
############################

# Importation de Seaborn
import seaborn as sns
import matplotlib.pyplot as plt

# Sélection des colonnes numériques pertinentes pour la corrélation
correlation_columns = [
    'stock_quantity',
    'total_sales',
    'price',
    'purchase_price',
    'price_ht',
    'taux_marge',
    'ca_par_article',
    'rotation_stock',
    'stock_value'
]

# Calcul de la matrice de corrélation
corr_matrix = df_marge[correlation_columns].corr()

# Création d'une heatmap de corrélation avec les variables de prix, de stock et de ventes
plt.figure(figsize=(8, 6))

sns.heatmap(corr_matrix,
    annot=True,
    cmap="Greens",
    fmt=".2f",
    linewidths=0.5,
    xticklabels=['Stock','Ventes','Prix','Prix achat','Prix HT','Taux de marge','CA','Rotation Stock','Valeur stock'],
    yticklabels=['Stock','Ventes','Prix','Prix achat','Prix HT','Taux de marge','CA','Rotation Stock','Valeur stock']
)

plt.title("Matrice de corrélation")
plt.show()

In [ ]:
# Création du mask (triangle supérieur) pour éviter les doublons visuels car la matrice est symétrique

mask = np.triu(np.ones_like(corr_matrix, dtype=bool)) # np.ones_like → crée une matrice de True,
                                                      # np.triu → garde le triangle supérieur
                                                      # mask → cache cette partie dans la heatmap

plt.figure(figsize=(8, 6))

sns.heatmap(corr_matrix,
    annot=True,
    mask = mask,
    cmap="Greens",
    fmt=".2f",
    linewidths=0.5,
    xticklabels=['Stock','Ventes','Prix','Prix achat','Prix HT','Taux de marge','CA','Rotation Stock','Valeur stock'],
    yticklabels=['Stock','Ventes','Prix','Prix achat','Prix HT','Taux de marge','CA','Rotation Stock','Valeur stock']
)

plt.title("Matrice de corrélation")
plt.show()

Stock vs Ventes (+0.44) ➡️ logique d’approvisionnement adaptée à la demande
Prix HT vs Ventes (-0.52) ➡️ Plus le prix augmente → moins les ventes sont élevées
Prix vs Prix d’achat (+0.98) ➡️ Le prix de vente est directement lié au prix d’achat
Prix d’achat vs Ventes (-0.50) ➡️ Les produits coûteux se vendent moins

<div style="border: 1px solid RGB(51,165,182);" >
<h3 style="margin: auto; padding: 20px; color: RGB(51,165,182); ">Etape 5.6 - Mise à disposition de la nouvelle table sur un fichier Excel</h3>
</div>

In [ ]:
# Export du dataset d'analyse sur un fichier Excel
df_analyse_final = df_marge.copy()
df_final.to_excel("df_final.xlsx", index=False)

In [ ]:
df_analyse_final.head()

In [ ]:
# Synthese des chiffres clés

ca_total = df_analyse_final['ca_par_article'].sum()
quantites_total = df_analyse_final['total_sales'].sum()
stock_total = df_analyse_final['stock_quantity'].sum()
valeur_stock = (df_analyse_final['stock_quantity'] * df_analyse_final['purchase_price']).sum()
print(f"Chiffre d'affaires total : {round(ca_total, 2)} €")
print(f"Quantités vendues : {int(quantites_total)}")
print(f"Stock total ERP : {int(stock_total)} unités")
print(f"Valeur de stock au prix d'achat : {round(valeur_stock, 2)} €")

Chiffre d'affaires total : 143204.7 €
Quantités vendues : 5726
Stock total ERP : 16678 unités
Valeur de stock au prix d'achat : 269240.57 €